In [1]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — DATA
# Pulls OMIE day-ahead prices via daily CSV scraper.
# Output: spot — pd.Series, hourly, DatetimeIndex (CET), €/MWh
# ═══════════════════════════════════════════════════════════════

import requests
import pandas as pd
import numpy as np
from datetime import date, timedelta
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────────
HISTORY_DAYS = 365        # how many calendar days to pull
END_DATE     = date.today() - timedelta(days=1)   # yesterday (latest available)
START_DATE   = END_DATE - timedelta(days=HISTORY_DAYS)
COUNTRY      = 'ES'       # ES = Spain, PT = Portugal


def fetch_omie_day(d: date) -> 'pd.Series | None':    """
    Fetches OMIE marginal price file for a single date.
    Returns 24-element pd.Series indexed by datetime (CET) or None on failure.
    OMIE column layout: date ; hour ; ES price ; PT price ; extra
    """
    date_str = d.strftime('%Y%m%d')
    url = (
        f'https://www.omie.es/es/file-access-list'
        f'?parents%5B%5D=/{d.year}/{d.month:02d}/{d.day:02d}/'
        f'&filename=marginalpdbc_{date_str}.1'
    )
    try:
        r = requests.get(url, timeout=15)
        if r.status_code != 200:
            return None
        lines = [l.strip() for l in r.text.splitlines() if l.strip()]
        # Skip header rows — data rows have exactly 5 semicolon-delimited fields
        rows = []
        for line in lines:
            parts = line.split(';')
            if len(parts) >= 4:
                try:
                    hour  = int(parts[1]) - 1          # OMIE is 1-indexed
                    price = float(parts[2].replace(',', '.'))
                    rows.append((hour, price))
                except (ValueError, IndexError):
                    continue
        if not rows:
            return None
        df_day = pd.DataFrame(rows, columns=['hour', 'price'])
        # Build DatetimeIndex: date + hour offset
        base = pd.Timestamp(d)
        df_day.index = df_day['hour'].apply(
            lambda h: base + pd.Timedelta(hours=h)
        )
        return df_day['price']
    except Exception:
        return None


# ── PULL ──────────────────────────────────────────────────────
print(f'Pulling OMIE daily files: {START_DATE} → {END_DATE}')
print('(one request per day — ~15s per 100 days)')

series_list = []
current     = START_DATE
fails       = 0

while current <= END_DATE:
    s = fetch_omie_day(current)
    if s is not None:
        series_list.append(s)
    else:
        fails += 1
    current += timedelta(days=1)

# ── ASSEMBLE ──────────────────────────────────────────────────
if not series_list:
    raise RuntimeError(
        'No data retrieved. Check network access to www.omie.es '
        'or run: curl https://www.omie.es from terminal.'
    )

spot = pd.concat(series_list).sort_index()
spot = spot[~spot.index.duplicated(keep='last')]   # drop any DST duplicates
spot.name = 'price_omie'

# ── VALIDATE ──────────────────────────────────────────────────
n_neg    = (spot < 0).sum()
n_rows   = len(spot)
coverage = n_rows / (HISTORY_DAYS * 24)

print()
print('─' * 45)
print(f'  Rows          : {n_rows:,} hourly observations')
print(f'  Period        : {spot.index[0].date()} → {spot.index[-1].date()}')
print(f'  Coverage      : {coverage:.1%} of expected hours')
print(f'  Negative hours: {n_neg:,}  ({n_neg/n_rows*100:.1f}%)')
print(f'  Mean price    : {spot.mean():.2f} €/MWh')
print(f'  Min price     : {spot.min():.2f} €/MWh')
print(f'  Max price     : {spot.max():.2f} €/MWh')
print(f'  Failed days   : {fails}')
print(f'  Last spot     : {spot.iloc[-1]:.2f} €/MWh')
print('─' * 45)

if coverage < 0.85:
    print('WARNING: coverage below 85% — check OMIE connectivity')
else:
    print('✓ Data ready')

/Users/oly/Documents/New project/peace-power-spain/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — DATA
# Pulls OMIE day-ahead prices via daily CSV scraper.
# Output: spot — pd.Series, hourly, DatetimeIndex (CET), €/MWh
# ═══════════════════════════════════════════════════════════════

import requests
import pandas as pd
import numpy as np
from datetime import date, timedelta
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────────
HISTORY_DAYS = 365        # how many calendar days to pull
END_DATE     = date.today() - timedelta(days=1)   # yesterday (latest available)
START_DATE   = END_DATE - timedelta(days=HISTORY_DAYS)
COUNTRY      = 'ES'       # ES = Spain, PT = Portugal


def fetch_omie_day(d: date) -> 'pd.Series | None':    """
    Fetches OMIE marginal price file for a single date.
    Returns 24-element pd.Series indexed by datetime (CET) or None on failure.
    OMIE column layout: date ; hour ; ES price ; PT price ; extra
    """
    date_str = d.strftime('%Y%m%d')
    url = (
        f'https://www.omie.es/es/file-access-list'
        f'?parents%5B%5D=/{d.year}/{d.month:02d}/{d.day:02d}/'
        f'&filename=marginalpdbc_{date_str}.1'
    )
    try:
        r = requests.get(url, timeout=15)
        if r.status_code != 200:
            return None
        lines = [l.strip() for l in r.text.splitlines() if l.strip()]
        # Skip header rows — data rows have exactly 5 semicolon-delimited fields
        rows = []
        for line in lines:
            parts = line.split(';')
            if len(parts) >= 4:
                try:
                    hour  = int(parts[1]) - 1          # OMIE is 1-indexed
                    price = float(parts[2].replace(',', '.'))
                    rows.append((hour, price))
                except (ValueError, IndexError):
                    continue
        if not rows:
            return None
        df_day = pd.DataFrame(rows, columns=['hour', 'price'])
        # Build DatetimeIndex: date + hour offset
        base = pd.Timestamp(d)
        df_day.index = df_day['hour'].apply(
            lambda h: base + pd.Timedelta(hours=h)
        )
        return df_day['price']
    except Exception:
        return None


# ── PULL ──────────────────────────────────────────────────────
print(f'Pulling OMIE daily files: {START_DATE} → {END_DATE}')
print('(one request per day — ~15s per 100 days)')

series_list = []
current     = START_DATE
fails       = 0

while current <= END_DATE:
    s = fetch_omie_day(current)
    if s is not None:
        series_list.append(s)
    else:
        fails += 1
    current += timedelta(days=1)

# ── ASSEMBLE ──────────────────────────────────────────────────
if not series_list:
    raise RuntimeError(
        'No data retrieved. Check network access to www.omie.es '
        'or run: curl https://www.omie.es from terminal.'
    )

spot = pd.concat(series_list).sort_index()
spot = spot[~spot.index.duplicated(keep='last')]   # drop any DST duplicates
spot.name = 'price_omie'

# ── VALIDATE ──────────────────────────────────────────────────
n_neg    = (spot < 0).sum()
n_rows   = len(spot)
coverage = n_rows / (HISTORY_DAYS * 24)

print()
print('─' * 45)
print(f'  Rows          : {n_rows:,} hourly observations')
print(f'  Period        : {spot.index[0].date()} → {spot.index[-1].date()}')
print(f'  Coverage      : {coverage:.1%} of expected hours')
print(f'  Negative hours: {n_neg:,}  ({n_neg/n_rows*100:.1f}%)')
print(f'  Mean price    : {spot.mean():.2f} €/MWh')
print(f'  Min price     : {spot.min():.2f} €/MWh')
print(f'  Max price     : {spot.max():.2f} €/MWh')
print(f'  Failed days   : {fails}')
print(f'  Last spot     : {spot.iloc[-1]:.2f} €/MWh')
print('─' * 45)

if coverage < 0.85:
    print('WARNING: coverage below 85% — check OMIE connectivity')
else:
    print('✓ Data ready')

IndentationError: unexpected indent (3282601272.py, line 26)

In [3]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — DATA
# Pulls OMIE day-ahead prices via daily CSV scraper.
# Output: spot — pd.Series, hourly, DatetimeIndex, €/MWh
# ═══════════════════════════════════════════════════════════════

import requests
import pandas as pd
import numpy as np
from datetime import date, timedelta
from typing import Optional
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────────
HISTORY_DAYS = 365
END_DATE     = date.today() - timedelta(days=1)
START_DATE   = END_DATE - timedelta(days=HISTORY_DAYS)


def fetch_omie_day(d):
    # type: (date) -> Optional[pd.Series]
    """
    Fetches OMIE marginal price file for a single date.
    Returns 24-element pd.Series indexed by datetime or None on failure.
    """
    date_str = d.strftime('%Y%m%d')
    url = (
        'https://www.omie.es/es/file-access-list'
        '?parents%5B%5D=/{}/{:02d}/{:02d}/'
        '&filename=marginalpdbc_{}.1'
    ).format(d.year, d.month, d.day, date_str)
    try:
        r = requests.get(url, timeout=15)
        if r.status_code != 200:
            return None
        lines = [l.strip() for l in r.text.splitlines() if l.strip()]
        rows = []
        for line in lines:
            parts = line.split(';')
            if len(parts) >= 4:
                try:
                    hour  = int(parts[1]) - 1
                    price = float(parts[2].replace(',', '.'))
                    rows.append((hour, price))
                except (ValueError, IndexError):
                    continue
        if not rows:
            return None
        df_day = pd.DataFrame(rows, columns=['hour', 'price'])
        base = pd.Timestamp(d)
        df_day.index = df_day['hour'].apply(
            lambda h: base + pd.Timedelta(hours=h)
        )
        return df_day['price']
    except Exception:
        return None


# ── PULL ──────────────────────────────────────────────────────
print('Pulling OMIE: {} to {}'.format(START_DATE, END_DATE))

series_list = []
current     = START_DATE
fails       = 0

while current <= END_DATE:
    s = fetch_omie_day(current)
    if s is not None:
        series_list.append(s)
    else:
        fails += 1
    current += timedelta(days=1)

if not series_list:
    raise RuntimeError(
        'No data retrieved. '
        'Test connectivity: curl https://www.omie.es from terminal.'
    )

spot = pd.concat(series_list).sort_index()
spot = spot[~spot.index.duplicated(keep='last')]
spot.name = 'price_omie'

# ── VALIDATE ──────────────────────────────────────────────────
n_neg    = int((spot < 0).sum())
n_rows   = len(spot)
coverage = n_rows / (HISTORY_DAYS * 24)

print('─' * 45)
print('  Rows          : {:,}'.format(n_rows))
print('  Period        : {} to {}'.format(spot.index[0].date(), spot.index[-1].date()))
print('  Coverage      : {:.1%}'.format(coverage))
print('  Negative hours: {:,}  ({:.1f}%)'.format(n_neg, n_neg / n_rows * 100))
print('  Mean price    : {:.2f} EUR/MWh'.format(spot.mean()))
print('  Min price     : {:.2f} EUR/MWh'.format(spot.min()))
print('  Max price     : {:.2f} EUR/MWh'.format(spot.max()))
print('  Failed days   : {}'.format(fails))
print('  Last spot     : {:.2f} EUR/MWh'.format(spot.iloc[-1]))
print('─' * 45)
print('OK Data ready')

Pulling OMIE: 2025-04-27 to 2026-04-27


RuntimeError: No data retrieved. Test connectivity: curl https://www.omie.es from terminal.

In [4]:
import requests
r = requests.get('https://www.omie.es', timeout=10)
print(r.status_code)

502


In [5]:
import requests
from datetime import date, timedelta

d = date.today() - timedelta(days=2)
date_str = d.strftime('%Y%m%d')
url = (
    'https://www.omie.es/es/file-access-list'
    '?parents%5B%5D=%2F{year}%2F{month:02d}%2F{day:02d}%2F'
    '&filename=marginalpdbc_{ds}.1'
).format(year=d.year, month=d.month, day=d.day, ds=date_str)

r = requests.get(url, timeout=15)
print(r.status_code)
print(r.text[:300])

502
<html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>Microsoft-Azure-Application-Gateway/v2</center>
</body>
</html>



In [6]:
import requests
from datetime import date, timedelta

d = date.today() - timedelta(days=2)
date_str = d.strftime('%Y%m%d')

# OMIE FTP mirror
url = 'https://www.omie.es/es/file-access-list?parents%5B%5D=%2F&parents%5B%5D=Precios%20del%20mercado%20diario&filename=marginalpdbc_{}.1'.format(date_str)

r = requests.get(url, timeout=15)
print(r.status_code)
print(r.text[:300])

502
<html>
<head><title>502 Bad Gateway</title></head>
<body>
<center><h1>502 Bad Gateway</h1></center>
<hr><center>Microsoft-Azure-Application-Gateway/v2</center>
</body>
</html>



In [7]:
r2 = requests.get('https://www.omie.es', timeout=10)
print(r2.status_code)

502


In [8]:
r2 = requests.get('https://www.omie.es', timeout=10)
print(r2.status_code)

502


In [9]:
import pickle
import pandas as pd

# Load the artifact you already have
with open('pc_spot_pricer_real.pkl', 'rb') as f:
    arts = pickle.load(f)

print(arts.keys())

# If spot is in there
spot = arts['spot']
print(spot.describe())
print('Last spot: {:.2f}'.format(spot.iloc[-1]))

dict_keys(['spot', 'pipeline_output'])
count    28674.000000
mean        68.568206
std         46.252301
min        -15.000000
25%         26.280000
50%         74.585000
75%        104.670000
max        240.000000
Name: price_omie, dtype: float64
Last spot: 82.00


In [10]:
import pickle

with open('pc_spot_pricer_real.pkl', 'rb') as f:
    arts = pickle.load(f)

spot = arts['spot']
print('Loaded: {:,} rows, last spot {:.2f} EUR/MWh'.format(len(spot), spot.iloc[-1]))

Loaded: 28,674 rows, last spot 82.00 EUR/MWh


In [11]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — FEATURES
# Vol, log returns, thermal floor proxy, spike probability.
# Negative price fix: shifted log-normal.
# ═══════════════════════════════════════════════════════════════

import numpy as np

# ── SHIFT ─────────────────────────────────────────────────────
# Moves all prices above zero before log transform.
# Removed again at payoff stage in Cell 3.
SHIFT        = abs(float(spot.min())) + 1.0
spot_shifted = spot + SHIFT

assert (spot_shifted > 0).all(), 'Shift failed — non-positive values remain'

# ── LOG RETURNS ───────────────────────────────────────────────
log_prices = np.log(spot_shifted)
returns    = log_prices.diff().dropna()

# ── ANNUALISED VOL ────────────────────────────────────────────
HOURS_PER_YEAR = 8760
vol_hourly     = float(returns.std())
vol_annual     = vol_hourly * np.sqrt(HOURS_PER_YEAR)

# ── LAST SPOT ─────────────────────────────────────────────────
last_spot = float(spot.iloc[-1])

# ── THERMAL FLOOR PROXY ───────────────────────────────────────
# Replace TTF_PROXY / EUA_PROXY with live values once ESIOS token is wired
TTF_PROXY      = 35.0
EUA_PROXY      = 65.0
HEAT_RATE      = 0.45
CO2_INTENS     = 0.35
floor_proxy    = TTF_PROXY * HEAT_RATE + EUA_PROXY * CO2_INTENS
price_vs_floor = last_spot - floor_proxy

# ── SPIKE PROBABILITY ─────────────────────────────────────────
rolling_mean = spot.rolling(24).mean()
rolling_std  = spot.rolling(24).std()
z_score      = (spot - rolling_mean) / rolling_std.clip(lower=0.01)
spike_prob   = float((z_score > 2.0).mean())

# ── NEGATIVE PRICE STATS ──────────────────────────────────────
neg_mask       = spot < 0
neg_price_prob = float(neg_mask.mean())
neg_price_mean = float(spot[neg_mask].mean()) if neg_mask.any() else 0.0

print('─' * 45)
print('  Feature Summary')
print('─' * 45)
print('  Last spot        : {:.2f} EUR/MWh'.format(last_spot))
print('  Shift constant   : {:.2f} EUR/MWh'.format(SHIFT))
print('  Shifted F0       : {:.2f} EUR/MWh'.format(last_spot + SHIFT))
print('  Annualised vol   : {:.4f}  ({:.2f}%)'.format(vol_annual, vol_annual * 100))
print('  Thermal floor    : {:.2f} EUR/MWh'.format(floor_proxy))
print('  Price vs floor   : {:+.2f} EUR/MWh'.format(price_vs_floor))
print('  Spike prob       : {:.4f}'.format(spike_prob))
print('  Neg. price prob  : {:.4f}'.format(neg_price_prob))
print('  Mean neg. price  : {:.2f} EUR/MWh'.format(neg_price_mean))
print('─' * 45)
print('OK Features ready')

─────────────────────────────────────────────
  Feature Summary
─────────────────────────────────────────────
  Last spot        : 82.00 EUR/MWh
  Shift constant   : 16.00 EUR/MWh
  Shifted F0       : 98.00 EUR/MWh
  Annualised vol   : 19.0888  (1908.88%)
  Thermal floor    : 38.50 EUR/MWh
  Price vs floor   : +43.50 EUR/MWh
  Spike prob       : 0.0360
  Neg. price prob  : 0.0320
  Mean neg. price  : -1.40 EUR/MWh
─────────────────────────────────────────────
OK Features ready


In [12]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — PRICING + SAVE
# Black-76 closed-form + Monte Carlo.
# Both operate on shifted forward. Shift removed at payoff.
# Saves pc_spot_pricer_real_v2.pkl
# ═══════════════════════════════════════════════════════════════

import pickle
import numpy as np
from scipy.stats import norm

# ── PARAMETERS ────────────────────────────────────────────────
T     = 1.0 / 12.0   # 1 month to expiry
r     = 0.03          # risk-free rate
K     = last_spot     # ATM strike
F     = last_spot     # forward = spot
sigma = vol_annual

F_s = F + SHIFT
K_s = K + SHIFT

assert F_s > 0,   'Shifted forward must be positive. Got {:.4f}'.format(F_s)
assert K_s > 0,   'Shifted strike must be positive. Got {:.4f}'.format(K_s)
assert sigma > 0, 'Vol must be positive. Got {:.4f}'.format(sigma)
assert T > 0,     'Expiry must be positive. Got {}'.format(T)


# ── BLACK-76 ──────────────────────────────────────────────────
def black76_call(F_s, K_s, sigma, T, r):
    sqrt_T = np.sqrt(T)
    d1 = (np.log(F_s / K_s) + 0.5 * sigma ** 2 * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T
    return float(np.exp(-r * T) * (F_s * norm.cdf(d1) - K_s * norm.cdf(d2)))


b76_call = black76_call(F_s, K_s, sigma, T, r)


# ── MONTE CARLO ───────────────────────────────────────────────
N_SIMS = 100000
np.random.seed(42)

Z        = np.random.standard_normal(N_SIMS)
ST_shift = F_s * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * Z)
ST_real  = ST_shift - SHIFT
payoffs  = np.maximum(ST_real - K, 0.0)
mc_call  = float(np.exp(-r * T) * payoffs.mean())
mc_se    = float(np.exp(-r * T) * payoffs.std() / np.sqrt(N_SIMS))


# ── RESULTS ───────────────────────────────────────────────────
print('=' * 45)
print('  PEACE CAPITAL -- OMIE Spot Pricer')
print('=' * 45)
print('  Last spot        : {:.2f} EUR/MWh'.format(last_spot))
print('  Annualised vol   : {:.2f}%'.format(vol_annual * 100))
print('  Shift constant   : {:.2f} EUR/MWh'.format(SHIFT))
print('─' * 45)
print('  Black-76 call    : {:.4f} EUR/MWh'.format(b76_call))
print('  MC call          : {:.4f} EUR/MWh  (+/- {:.4f} SE)'.format(mc_call, mc_se))
print('  B76 vs MC diff   : {:.4f} EUR/MWh'.format(abs(b76_call - mc_call)))
print('─' * 45)
print('  Thermal floor    : {:.2f} EUR/MWh'.format(floor_proxy))
print('  Price vs floor   : {:+.2f} EUR/MWh'.format(price_vs_floor))
print('  Spike prob       : {:.4f}'.format(spike_prob))
print('  Neg. price prob  : {:.4f}'.format(neg_price_prob))
print('=' * 45)


# ── SAVE ──────────────────────────────────────────────────────
artifact = {
    'last_spot':         last_spot,
    'annualised_vol':    vol_annual,
    'black76_call':      b76_call,
    'mc_call':           mc_call,
    'mc_se':             mc_se,
    'spike_prob':        spike_prob,
    'neg_price_prob':    neg_price_prob,
    'neg_price_mean':    neg_price_mean,
    'strike':            K,
    'forward':           F,
    'shift_constant':    SHIFT,
    'floor_proxy':       floor_proxy,
    'price_vs_floor':    price_vs_floor,
    'T':                 T,
    'r':                 r,
    'sigma':             sigma,
    'data_start':        str(spot.index[0].date()),
    'data_end':          str(spot.index[-1].date()),
    'n_hours':           len(spot),
    'n_negative_hours':  int((spot < 0).sum()),
    'spot':              spot,
    'returns':           returns,
}

PKL_PATH = 'pc_spot_pricer_real_v2.pkl'
with open(PKL_PATH, 'wb') as f:
    pickle.dump(artifact, f)

print('\nOK Saved to {}'.format(PKL_PATH))

# ── VERIFY ────────────────────────────────────────────────────
with open(PKL_PATH, 'rb') as f:
    check = pickle.load(f)

assert abs(check['last_spot']      - last_spot)  < 0.01
assert abs(check['annualised_vol'] - vol_annual) < 0.001
assert abs(check['mc_call']        - mc_call)    < 0.001
print('OK Reload verified -- artifact is clean')
print('Keys: {}'.format(list(check.keys())))

  PEACE CAPITAL -- OMIE Spot Pricer
  Last spot        : 82.00 EUR/MWh
  Annualised vol   : 1908.88%
  Shift constant   : 16.00 EUR/MWh
─────────────────────────────────────────────
  Black-76 call    : 97.1820 EUR/MWh
  MC call          : 19.2396 EUR/MWh  (+/- 13.1430 SE)
  B76 vs MC diff   : 77.9423 EUR/MWh
─────────────────────────────────────────────
  Thermal floor    : 38.50 EUR/MWh
  Price vs floor   : +43.50 EUR/MWh
  Spike prob       : 0.0360
  Neg. price prob  : 0.0320

OK Saved to pc_spot_pricer_real_v2.pkl
OK Reload verified -- artifact is clean
Keys: ['last_spot', 'annualised_vol', 'black76_call', 'mc_call', 'mc_se', 'spike_prob', 'neg_price_prob', 'neg_price_mean', 'strike', 'forward', 'shift_constant', 'floor_proxy', 'price_vs_floor', 'T', 'r', 'sigma', 'data_start', 'data_end', 'n_hours', 'n_negative_hours', 'spot', 'returns']


In [13]:
# What is the actual frequency of your spot series?
print(spot.index[:5])
print(spot.index[1] - spot.index[0])  # time delta between observations
print(len(spot))

DatetimeIndex(['2023-01-01 00:00:00+01:00', '2023-01-01 01:00:00+01:00',
               '2023-01-01 02:00:00+01:00', '2023-01-01 03:00:00+01:00',
               '2023-01-01 04:00:00+01:00'],
              dtype='datetime64[ns, Europe/Madrid]', name='datetime', freq=None)
0 days 01:00:00
28674


In [14]:
# Replace this line in Cell 2:
HOURS_PER_YEAR = 8760

# With whichever applies:
HOURS_PER_YEAR = 365   # if daily data
HOURS_PER_YEAR = 8760  # if hourly data

In [15]:
# ── VOL: computed on unshifted prices, excluding negative hours ──
# Shift is only for log transform in MC/B76 — not for vol estimation
spot_pos    = spot[spot > 0]
log_pos     = np.log(spot_pos)
returns_pos = log_pos.diff().dropna()

# Drop extreme outliers (beyond 4 std) before vol estimation
z           = (returns_pos - returns_pos.mean()) / returns_pos.std()
returns_clean = returns_pos[z.abs() < 4]

vol_hourly  = float(returns_clean.std())
vol_annual  = vol_hourly * np.sqrt(HOURS_PER_YEAR)

# Keep full returns series (with negatives, shifted) for other uses
log_prices  = np.log(spot_shifted)
returns     = log_prices.diff().dropna()

In [16]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — FEATURES
# Vol, log returns, thermal floor proxy, spike probability.
# Negative price fix: shifted log-normal.
# ═══════════════════════════════════════════════════════════════

import numpy as np

# ── SHIFT ─────────────────────────────────────────────────────
# Moves all prices above zero before log transform.
# Removed again at payoff stage in Cell 3.
SHIFT        = abs(float(spot.min())) + 1.0
spot_shifted = spot + SHIFT

assert (spot_shifted > 0).all(), 'Shift failed — non-positive values remain'

# ── LOG RETURNS ───────────────────────────────────────────────
log_prices = np.log(spot_shifted)
returns    = log_prices.diff().dropna()

# ── ANNUALISED VOL ────────────────────────────────────────────
HOURS_PER_YEAR = 8760
vol_hourly     = float(returns.std())
vol_annual     = vol_hourly * np.sqrt(HOURS_PER_YEAR)

# ── LAST SPOT ─────────────────────────────────────────────────
last_spot = float(spot.iloc[-1])

# ── THERMAL FLOOR PROXY ───────────────────────────────────────
# Replace TTF_PROXY / EUA_PROXY with live values once ESIOS token is wired
TTF_PROXY      = 35.0
EUA_PROXY      = 65.0
HEAT_RATE      = 0.45
CO2_INTENS     = 0.35
floor_proxy    = TTF_PROXY * HEAT_RATE + EUA_PROXY * CO2_INTENS
price_vs_floor = last_spot - floor_proxy

# ── SPIKE PROBABILITY ─────────────────────────────────────────
rolling_mean = spot.rolling(24).mean()
rolling_std  = spot.rolling(24).std()
z_score      = (spot - rolling_mean) / rolling_std.clip(lower=0.01)
spike_prob   = float((z_score > 2.0).mean())

# ── NEGATIVE PRICE STATS ──────────────────────────────────────
neg_mask       = spot < 0
neg_price_prob = float(neg_mask.mean())
neg_price_mean = float(spot[neg_mask].mean()) if neg_mask.any() else 0.0

print('─' * 45)
print('  Feature Summary')
print('─' * 45)
print('  Last spot        : {:.2f} EUR/MWh'.format(last_spot))
print('  Shift constant   : {:.2f} EUR/MWh'.format(SHIFT))
print('  Shifted F0       : {:.2f} EUR/MWh'.format(last_spot + SHIFT))
print('  Annualised vol   : {:.4f}  ({:.2f}%)'.format(vol_annual, vol_annual * 100))
print('  Thermal floor    : {:.2f} EUR/MWh'.format(floor_proxy))
print('  Price vs floor   : {:+.2f} EUR/MWh'.format(price_vs_floor))
print('  Spike prob       : {:.4f}'.format(spike_prob))
print('  Neg. price prob  : {:.4f}'.format(neg_price_prob))
print('  Mean neg. price  : {:.2f} EUR/MWh'.format(neg_price_mean))
print('─' * 45)
print('OK Features ready')

─────────────────────────────────────────────
  Feature Summary
─────────────────────────────────────────────
  Last spot        : 82.00 EUR/MWh
  Shift constant   : 16.00 EUR/MWh
  Shifted F0       : 98.00 EUR/MWh
  Annualised vol   : 19.0888  (1908.88%)
  Thermal floor    : 38.50 EUR/MWh
  Price vs floor   : +43.50 EUR/MWh
  Spike prob       : 0.0360
  Neg. price prob  : 0.0320
  Mean neg. price  : -1.40 EUR/MWh
─────────────────────────────────────────────
OK Features ready


In [17]:
 ═══════════════════════════════════════════════════════════════
# CELL 3 — PRICING + SAVE
# Black-76 closed-form + Monte Carlo.
# Both operate on shifted forward. Shift removed at payoff.
# Saves pc_spot_pricer_real_v2.pkl
# ═══════════════════════════════════════════════════════════════

import pickle
import numpy as np
from scipy.stats import norm

# ── PARAMETERS ────────────────────────────────────────────────
T     = 1.0 / 12.0   # 1 month to expiry
r     = 0.03          # risk-free rate
K     = last_spot     # ATM strike
F     = last_spot     # forward = spot
sigma = vol_annual

F_s = F + SHIFT
K_s = K + SHIFT

assert F_s > 0,   'Shifted forward must be positive. Got {:.4f}'.format(F_s)
assert K_s > 0,   'Shifted strike must be positive. Got {:.4f}'.format(K_s)
assert sigma > 0, 'Vol must be positive. Got {:.4f}'.format(sigma)
assert T > 0,     'Expiry must be positive. Got {}'.format(T)


# ── BLACK-76 ──────────────────────────────────────────────────
def black76_call(F_s, K_s, sigma, T, r):
    sqrt_T = np.sqrt(T)
    d1 = (np.log(F_s / K_s) + 0.5 * sigma ** 2 * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T
    return float(np.exp(-r * T) * (F_s * norm.cdf(d1) - K_s * norm.cdf(d2)))


b76_call = black76_call(F_s, K_s, sigma, T, r)


# ── MONTE CARLO ───────────────────────────────────────────────
N_SIMS = 100000
np.random.seed(42)

Z        = np.random.standard_normal(N_SIMS)
ST_shift = F_s * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * Z)
ST_real  = ST_shift - SHIFT
payoffs  = np.maximum(ST_real - K, 0.0)
mc_call  = float(np.exp(-r * T) * payoffs.mean())
mc_se    = float(np.exp(-r * T) * payoffs.std() / np.sqrt(N_SIMS))


# ── RESULTS ───────────────────────────────────────────────────
print('=' * 45)
print('  PEACE CAPITAL -- OMIE Spot Pricer')
print('=' * 45)
print('  Last spot        : {:.2f} EUR/MWh'.format(last_spot))
print('  Annualised vol   : {:.2f}%'.format(vol_annual * 100))
print('  Shift constant   : {:.2f} EUR/MWh'.format(SHIFT))
print('─' * 45)
print('  Black-76 call    : {:.4f} EUR/MWh'.format(b76_call))
print('  MC call          : {:.4f} EUR/MWh  (+/- {:.4f} SE)'.format(mc_call, mc_se))
print('  B76 vs MC diff   : {:.4f} EUR/MWh'.format(abs(b76_call - mc_call)))
print('─' * 45)
print('  Thermal floor    : {:.2f} EUR/MWh'.format(floor_proxy))
print('  Price vs floor   : {:+.2f} EUR/MWh'.format(price_vs_floor))
print('  Spike prob       : {:.4f}'.format(spike_prob))
print('  Neg. price prob  : {:.4f}'.format(neg_price_prob))
print('=' * 45)


# ── SAVE ──────────────────────────────────────────────────────
artifact = {
    'last_spot':         last_spot,
    'annualised_vol':    vol_annual,
    'black76_call':      b76_call,
    'mc_call':           mc_call,
    'mc_se':             mc_se,
    'spike_prob':        spike_prob,
    'neg_price_prob':    neg_price_prob,
    'neg_price_mean':    neg_price_mean,
    'strike':            K,
    'forward':           F,
    'shift_constant':    SHIFT,
    'floor_proxy':       floor_proxy,
    'price_vs_floor':    price_vs_floor,
    'T':                 T,
    'r':                 r,
    'sigma':             sigma,
    'data_start':        str(spot.index[0].date()),
    'data_end':          str(spot.index[-1].date()),
    'n_hours':           len(spot),
    'n_negative_hours':  int((spot < 0).sum()),
    'spot':              spot,
    'returns':           returns,
}

PKL_PATH = 'pc_spot_pricer_real_v2.pkl'
with open(PKL_PATH, 'wb') as f:
    pickle.dump(artifact, f)

print('\nOK Saved to {}'.format(PKL_PATH))

# ── VERIFY ────────────────────────────────────────────────────
with open(PKL_PATH, 'rb') as f:
    check = pickle.load(f)

assert abs(check['last_spot']      - last_spot)  < 0.01
assert abs(check['annualised_vol'] - vol_annual) < 0.001
assert abs(check['mc_call']        - mc_call)    < 0.001
print('OK Reload verified -- artifact is clean')
print('Keys: {}'.format(list(check.keys())))

SyntaxError: invalid character '═' (U+2550) (1375956927.py, line 1)

In [18]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — PRICING + SAVE
# Black-76 closed-form + Monte Carlo.
# Both operate on shifted forward. Shift removed at payoff.
# Saves pc_spot_pricer_real_v2.pkl
# ═══════════════════════════════════════════════════════════════

import pickle
import numpy as np
from scipy.stats import norm

# ── PARAMETERS ────────────────────────────────────────────────
T     = 1.0 / 12.0   # 1 month to expiry
r     = 0.03          # risk-free rate
K     = last_spot     # ATM strike
F     = last_spot     # forward = spot
sigma = vol_annual

F_s = F + SHIFT
K_s = K + SHIFT

assert F_s > 0,   'Shifted forward must be positive. Got {:.4f}'.format(F_s)
assert K_s > 0,   'Shifted strike must be positive. Got {:.4f}'.format(K_s)
assert sigma > 0, 'Vol must be positive. Got {:.4f}'.format(sigma)
assert T > 0,     'Expiry must be positive. Got {}'.format(T)


# ── BLACK-76 ──────────────────────────────────────────────────
def black76_call(F_s, K_s, sigma, T, r):
    sqrt_T = np.sqrt(T)
    d1 = (np.log(F_s / K_s) + 0.5 * sigma ** 2 * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T
    return float(np.exp(-r * T) * (F_s * norm.cdf(d1) - K_s * norm.cdf(d2)))


b76_call = black76_call(F_s, K_s, sigma, T, r)


# ── MONTE CARLO ───────────────────────────────────────────────
N_SIMS = 100000
np.random.seed(42)

Z        = np.random.standard_normal(N_SIMS)
ST_shift = F_s * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * Z)
ST_real  = ST_shift - SHIFT
payoffs  = np.maximum(ST_real - K, 0.0)
mc_call  = float(np.exp(-r * T) * payoffs.mean())
mc_se    = float(np.exp(-r * T) * payoffs.std() / np.sqrt(N_SIMS))


# ── RESULTS ───────────────────────────────────────────────────
print('=' * 45)
print('  PEACE CAPITAL -- OMIE Spot Pricer')
print('=' * 45)
print('  Last spot        : {:.2f} EUR/MWh'.format(last_spot))
print('  Annualised vol   : {:.2f}%'.format(vol_annual * 100))
print('  Shift constant   : {:.2f} EUR/MWh'.format(SHIFT))
print('─' * 45)
print('  Black-76 call    : {:.4f} EUR/MWh'.format(b76_call))
print('  MC call          : {:.4f} EUR/MWh  (+/- {:.4f} SE)'.format(mc_call, mc_se))
print('  B76 vs MC diff   : {:.4f} EUR/MWh'.format(abs(b76_call - mc_call)))
print('─' * 45)
print('  Thermal floor    : {:.2f} EUR/MWh'.format(floor_proxy))
print('  Price vs floor   : {:+.2f} EUR/MWh'.format(price_vs_floor))
print('  Spike prob       : {:.4f}'.format(spike_prob))
print('  Neg. price prob  : {:.4f}'.format(neg_price_prob))
print('=' * 45)


# ── SAVE ──────────────────────────────────────────────────────
artifact = {
    'last_spot':         last_spot,
    'annualised_vol':    vol_annual,
    'black76_call':      b76_call,
    'mc_call':           mc_call,
    'mc_se':             mc_se,
    'spike_prob':        spike_prob,
    'neg_price_prob':    neg_price_prob,
    'neg_price_mean':    neg_price_mean,
    'strike':            K,
    'forward':           F,
    'shift_constant':    SHIFT,
    'floor_proxy':       floor_proxy,
    'price_vs_floor':    price_vs_floor,
    'T':                 T,
    'r':                 r,
    'sigma':             sigma,
    'data_start':        str(spot.index[0].date()),
    'data_end':          str(spot.index[-1].date()),
    'n_hours':           len(spot),
    'n_negative_hours':  int((spot < 0).sum()),
    'spot':              spot,
    'returns':           returns,
}

PKL_PATH = 'pc_spot_pricer_real_v2.pkl'
with open(PKL_PATH, 'wb') as f:
    pickle.dump(artifact, f)

print('\nOK Saved to {}'.format(PKL_PATH))

# ── VERIFY ────────────────────────────────────────────────────
with open(PKL_PATH, 'rb') as f:
    check = pickle.load(f)

assert abs(check['last_spot']      - last_spot)  < 0.01
assert abs(check['annualised_vol'] - vol_annual) < 0.001
assert abs(check['mc_call']        - mc_call)    < 0.001
print('OK Reload verified -- artifact is clean')
print('Keys: {}'.format(list(check.keys())))

  PEACE CAPITAL -- OMIE Spot Pricer
  Last spot        : 82.00 EUR/MWh
  Annualised vol   : 1908.88%
  Shift constant   : 16.00 EUR/MWh
─────────────────────────────────────────────
  Black-76 call    : 97.1820 EUR/MWh
  MC call          : 19.2396 EUR/MWh  (+/- 13.1430 SE)
  B76 vs MC diff   : 77.9423 EUR/MWh
─────────────────────────────────────────────
  Thermal floor    : 38.50 EUR/MWh
  Price vs floor   : +43.50 EUR/MWh
  Spike prob       : 0.0360
  Neg. price prob  : 0.0320

OK Saved to pc_spot_pricer_real_v2.pkl
OK Reload verified -- artifact is clean
Keys: ['last_spot', 'annualised_vol', 'black76_call', 'mc_call', 'mc_se', 'spike_prob', 'neg_price_prob', 'neg_price_mean', 'strike', 'forward', 'shift_constant', 'floor_proxy', 'price_vs_floor', 'T', 'r', 'sigma', 'data_start', 'data_end', 'n_hours', 'n_negative_hours', 'spot', 'returns']


In [1]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — LOAD DATA
# Loads existing spot series from pickle.
# OMIE is currently returning 502 — use cached data.
# Replace this cell with the live scraper once OMIE is back up.
# ═══════════════════════════════════════════════════════════════

import pickle
import pandas as pd
import numpy as np
from typing import Optional
import warnings
warnings.filterwarnings('ignore')

PKL_SOURCE = 'pc_spot_pricer_real.pkl'

with open(PKL_SOURCE, 'rb') as f:
    arts = pickle.load(f)

spot = arts['spot']
spot.name = 'price_omie'

# ── VALIDATE ──────────────────────────────────────────────────
n_neg    = int((spot < 0).sum())
n_rows   = len(spot)
freq     = spot.index[1] - spot.index[0]

print('─' * 45)
print('  Source        : {}'.format(PKL_SOURCE))
print('  Rows          : {:,}'.format(n_rows))
print('  Frequency     : {}'.format(freq))
print('  Period        : {} to {}'.format(spot.index[0].date(), spot.index[-1].date()))
print('  Negative hours: {:,}  ({:.1f}%)'.format(n_neg, n_neg / n_rows * 100))
print('  Mean price    : {:.2f} EUR/MWh'.format(spot.mean()))
print('  Min price     : {:.2f} EUR/MWh'.format(spot.min()))
print('  Max price     : {:.2f} EUR/MWh'.format(spot.max()))
print('  Last spot     : {:.2f} EUR/MWh'.format(spot.iloc[-1]))
print('─' * 45)
print('OK Data loaded')

─────────────────────────────────────────────
  Source        : pc_spot_pricer_real.pkl
  Rows          : 28,674
  Frequency     : 0 days 01:00:00
  Period        : 2023-01-01 to 2026-04-09
  Negative hours: 917  (3.2%)
  Mean price    : 68.57 EUR/MWh
  Min price     : -15.00 EUR/MWh
  Max price     : 240.00 EUR/MWh
  Last spot     : 82.00 EUR/MWh
─────────────────────────────────────────────
OK Data loaded


In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — FEATURES
#
# Vol is estimated on UNSHIFTED positive-price returns only.
# Outliers beyond 4 sigma are clipped before estimation.
# This prevents the shift transformation from inflating vol.
#
# The SHIFT is only used in Cells 3 for log-normal pricing.
# ═══════════════════════════════════════════════════════════════

HOURS_PER_YEAR = 8760

# ── STEP 1: Vol on positive unshifted prices ──────────────────
spot_pos      = spot[spot > 0]
log_pos       = np.log(spot_pos)
returns_pos   = log_pos.diff().dropna()

# Clip outliers beyond 4 sigma
z_pos         = (returns_pos - returns_pos.mean()) / returns_pos.std()
returns_clean = returns_pos[z_pos.abs() < 4]

vol_hourly    = float(returns_clean.std())
vol_annual    = vol_hourly * np.sqrt(HOURS_PER_YEAR)

# ── STEP 2: Shift for MC/B76 (separate from vol) ─────────────
SHIFT        = abs(float(spot.min())) + 1.0
spot_shifted = spot + SHIFT

assert (spot_shifted > 0).all(), 'Shift failed — non-positive values remain'

# Full shifted returns kept for reference
log_shifted = np.log(spot_shifted)
returns     = log_shifted.diff().dropna()

# ── STEP 3: Last spot and derived features ────────────────────
last_spot = float(spot.iloc[-1])

# Thermal floor proxy — update with live TTF/EUA once ESIOS token is wired
TTF_PROXY      = 35.0
EUA_PROXY      = 65.0
HEAT_RATE      = 0.45
CO2_INTENS     = 0.35
floor_proxy    = TTF_PROXY * HEAT_RATE + EUA_PROXY * CO2_INTENS
price_vs_floor = last_spot - floor_proxy

# Spike probability: fraction of hours > 2 sigma above 24h rolling mean
rolling_mean = spot.rolling(24).mean()
rolling_std  = spot.rolling(24).std()
z_score      = (spot - rolling_mean) / rolling_std.clip(lower=0.01)
spike_prob   = float((z_score > 2.0).mean())

# Negative price stats
neg_mask       = spot < 0
neg_price_prob = float(neg_mask.mean())
neg_price_mean = float(spot[neg_mask].mean()) if neg_mask.any() else 0.0

print('─' * 45)
print('  Feature Summary')
print('─' * 45)
print('  Last spot        : {:.2f} EUR/MWh'.format(last_spot))
print('  Shift constant   : {:.2f} EUR/MWh'.format(SHIFT))
print('  Shifted F0       : {:.2f} EUR/MWh'.format(last_spot + SHIFT))
print('  Positive obs used: {:,} of {:,}'.format(len(returns_clean), len(spot)))
print('  Outliers clipped : {:,}'.format(len(returns_pos) - len(returns_clean)))
print('  Vol hourly       : {:.6f}'.format(vol_hourly))
print('  Annualised vol   : {:.4f}  ({:.2f}%)'.format(vol_annual, vol_annual * 100))
print('  Thermal floor    : {:.2f} EUR/MWh'.format(floor_proxy))
print('  Price vs floor   : {:+.2f} EUR/MWh'.format(price_vs_floor))
print('  Spike prob       : {:.4f}'.format(spike_prob))
print('  Neg. price prob  : {:.4f}'.format(neg_price_prob))
print('  Mean neg. price  : {:.2f} EUR/MWh'.format(neg_price_mean))
print('─' * 45)
print('OK Features ready')

─────────────────────────────────────────────
  Feature Summary
─────────────────────────────────────────────
  Last spot        : 82.00 EUR/MWh
  Shift constant   : 16.00 EUR/MWh
  Shifted F0       : 98.00 EUR/MWh
  Positive obs used: 26,368 of 28,674
  Outliers clipped : 376
  Vol hourly       : 0.406206
  Annualised vol   : 38.0188  (3801.88%)
  Thermal floor    : 38.50 EUR/MWh
  Price vs floor   : +43.50 EUR/MWh
  Spike prob       : 0.0360
  Neg. price prob  : 0.0320
  Mean neg. price  : -1.40 EUR/MWh
─────────────────────────────────────────────
OK Features ready


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — PRICING
# Black-76 closed-form + Monte Carlo.
# Both operate on shifted forward/strike.
# Shift is removed at payoff calculation.
# ═══════════════════════════════════════════════════════════════

from scipy.stats import norm

# ── PARAMETERS ────────────────────────────────────────────────
T     = 1.0 / 12.0   # 1 month to expiry
r     = 0.03          # risk-free rate (ECB proxy)
K     = last_spot     # ATM strike
F     = last_spot     # forward = spot
sigma = vol_annual    # vol from Cell 2 clean estimation

# Shifted inputs for log-normal pricing
F_s = F + SHIFT
K_s = K + SHIFT

# Hard assertions before pricing runs
assert F_s   > 0, 'Shifted forward must be positive. Got {:.4f}'.format(F_s)
assert K_s   > 0, 'Shifted strike must be positive. Got {:.4f}'.format(K_s)
assert sigma > 0, 'Vol must be positive. Got {:.6f}'.format(sigma)
assert T     > 0, 'Expiry must be positive. Got {}'.format(T)
assert sigma < 5, 'Vol above 500% — check vol estimation. Got {:.2f}'.format(sigma)


# ── BLACK-76 ──────────────────────────────────────────────────
def black76_call(F_s, K_s, sigma, T, r):
    sqrt_T = np.sqrt(T)
    d1 = (np.log(F_s / K_s) + 0.5 * sigma ** 2 * T) / (sigma * sqrt_T)
    d2 = d1 - sigma * sqrt_T
    return float(np.exp(-r * T) * (F_s * norm.cdf(d1) - K_s * norm.cdf(d2)))


b76_call = black76_call(F_s, K_s, sigma, T, r)


# ── MONTE CARLO ───────────────────────────────────────────────
N_SIMS = 100000
np.random.seed(42)

Z        = np.random.standard_normal(N_SIMS)
ST_shift = F_s * np.exp((r - 0.5 * sigma ** 2) * T + sigma * np.sqrt(T) * Z)
ST_real  = ST_shift - SHIFT          # unshift back to real price space
payoffs  = np.maximum(ST_real - K, 0.0)
mc_call  = float(np.exp(-r * T) * payoffs.mean())
mc_se    = float(np.exp(-r * T) * payoffs.std() / np.sqrt(N_SIMS))


# ── RESULTS ───────────────────────────────────────────────────
print('=' * 45)
print('  PEACE CAPITAL -- OMIE Spot Pricer')
print('=' * 45)
print('  Last spot        : {:.2f} EUR/MWh'.format(last_spot))
print('  Annualised vol   : {:.2f}%'.format(vol_annual * 100))
print('  Shift constant   : {:.2f} EUR/MWh'.format(SHIFT))
print('─' * 45)
print('  Black-76 call    : {:.4f} EUR/MWh'.format(b76_call))
print('  MC call          : {:.4f} EUR/MWh  (+/- {:.4f} SE)'.format(mc_call, mc_se))
print('  B76 vs MC diff   : {:.4f} EUR/MWh'.format(abs(b76_call - mc_call)))
print('─' * 45)
print('  Thermal floor    : {:.2f} EUR/MWh'.format(floor_proxy))
print('  Price vs floor   : {:+.2f} EUR/MWh'.format(price_vs_floor))
print('  Spike prob       : {:.4f}'.format(spike_prob))
print('  Neg. price prob  : {:.4f}'.format(neg_price_prob))
print('=' * 45)

AssertionError: Vol above 500% — check vol estimation. Got 38.02